Este notebook documenta el flujo NFL (New Football Learning) aplicado al dataset de activos de red.
El objetivo es generar un EDA completo, proponer ingeniería de características y dejar el dataset listo para entrenar un modelo DL enfocado en la métrica UITI.


## Flujo NFL propuesto
1. Preparar dependencias e importar librerías clave.
2. Cargar el dataset exacto mediante tu snippet kagglehub y validar su forma.
3. Ejecutar un EDA guiado (estructura, nulos, distribuciones, correlaciones, frecuencia de categorías).
4. Diseñar ingeniería de características útil para UITI y activos eléctricos.
5. Construir el pipeline de preprocesamiento y los conjuntos `train/valid` listos para DL.
6. Crear utilidades de `tf.data` y la plantilla de red neuronal para acelerar el entrenamiento NFL.


In [ ]:
# -----------------------------------------------------------
# Configuración general del entorno: warnings, librerías y estilo gráfico
# -----------------------------------------------------------
# %pip install --quiet kagglehub[pandas-datasets] pandas numpy matplotlib seaborn plotly scikit-learn tensorflow

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

from pathlib import Path
from typing import List, Sequence

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, StandardScaler

try:
    import tensorflow as tf
except ImportError:
    tf = None
    print('TensorFlow no está instalado; instala tensorflow si deseas ejecutar la parte DL.')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)


In [ ]:
# -----------------------------------------------------------
# Descarga oficial del dataset usando tu snippet kagglehub (mantener estilo)
# -----------------------------------------------------------
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "dataset_modificado.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "cristiancamiloo/powergrid-assets-ml-dataset",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

# Variables globales del flujo NFL
TARGET_COL = 'UITI'
df_original = df.copy()
print(f'Dataset con {df.shape[0]:,} filas y {df.shape[1]:,} columnas')


In [ ]:
# -----------------------------------------------------------
# Muestreos rápidos para validar consistencia antes de la EDA
# -----------------------------------------------------------
with pd.option_context('display.max_columns', None):
    display(df.head(3))
    display(df.tail(3))
    display(df.sample(min(5, len(df)), random_state=42))


In [ ]:
# -----------------------------------------------------------
# Métricas cuantitativas básicas del dataset original
# -----------------------------------------------------------
overview = pd.DataFrame(
    {
        'metricas': ['total_filas', 'total_columnas', 'duplicados', 'memoria_MB'],
        'valor': [
            int(len(df)),
            int(df.shape[1]),
            int(df.duplicated().sum()),
            round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2),
        ],
    }
)
display(overview)


In [ ]:
# -----------------------------------------------------------
# Reporte estructural: tipos, nulos y cardinalidad
# -----------------------------------------------------------
structure_report = pd.DataFrame(
    {
        'columna': df.columns,
        'tipo_python': df.dtypes.astype(str).values,
        'nulos': df.isna().sum().values,
        'pct_nulos': (df.isna().sum().values / len(df)),
        'cardinalidad': df.nunique(dropna=False).values,
    }
).sort_values(by='pct_nulos', ascending=False)

display(structure_report)


In [ ]:
# -----------------------------------------------------------
# Clasificación de variables (numéricas, categóricas, fechas, ids)
# -----------------------------------------------------------
assert TARGET_COL in df.columns, f'La columna objetivo {TARGET_COL} no existe.'

numeric_features = [col for col in df.select_dtypes(include=[np.number]).columns if col != TARGET_COL]
categorical_candidates = [col for col in df.select_dtypes(exclude=[np.number]).columns if col != TARGET_COL]


def detect_datetime_columns(dataframe: pd.DataFrame, candidates: List[str], min_coverage: float = 0.65):
    datetime_cols = []
    coverage_map = {}
    for col in candidates:
        parsed = pd.to_datetime(dataframe[col], errors='coerce', infer_datetime_format=True)
        coverage = parsed.notna().mean()
        coverage_map[col] = round(float(coverage), 3)
        if coverage >= min_coverage:
            datetime_cols.append(col)
    return datetime_cols, coverage_map


datetime_features, datetime_coverage = detect_datetime_columns(df, categorical_candidates)
categorical_features = [col for col in categorical_candidates if col not in datetime_features]

id_like_features = [
    col
    for col in df.columns
    if any(token in col.lower() for token in ['id', 'codigo', 'cod_', 'asset', 'equipo', 'point'])
]

feature_catalog = pd.DataFrame(
    {
        'grupo': ['target', 'numeric', 'categorical', 'datetime', 'id_like'],
        'total_columnas': [
            1,
            len(numeric_features),
            len(categorical_features),
            len(datetime_features),
            len(id_like_features),
        ],
        'ejemplos': [
            TARGET_COL,
            ', '.join(numeric_features[:5]) or '-',
            ', '.join(categorical_features[:5]) or '-',
            ', '.join(datetime_features[:5]) or '-',
            ', '.join(id_like_features[:5]) or '-',
        ],
    }
)

print('Cobertura detectada para columnas datetime:', datetime_coverage)
display(feature_catalog)


In [ ]:
# -----------------------------------------------------------
# Mapa de valores faltantes priorizando imputaciones
# -----------------------------------------------------------
missing_report = (
    df.isna()
    .sum()
    .to_frame(name='nulos')
    .assign(pct=lambda x: x['nulos'] / len(df))
    .sort_values(by='pct', ascending=False)
)
missing_report = missing_report[missing_report['nulos'] > 0]

if missing_report.empty:
    print('No se detectaron nulos.')
else:
    display(missing_report)
    px.bar(
        missing_report.reset_index().rename(columns={'index': 'columna'}),
        x='columna',
        y='pct',
        title='Porcentaje de valores faltantes',
    ).show()


In [ ]:
# -----------------------------------------------------------
# Estadísticas descriptivas para numericas + target
# -----------------------------------------------------------
if numeric_features:
    summary_numeric = df[numeric_features + [TARGET_COL]].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
    display(summary_numeric)
else:
    print('No existan variables numéricas aparte de la meta.')


In [ ]:
# -----------------------------------------------------------
# Distribuciones y sesgos de variables numéricas relevantes
# -----------------------------------------------------------
if numeric_features:
    selected_numeric = (
        df[numeric_features]
        .std()
        .sort_values(ascending=False)
        .head(min(6, len(numeric_features)))
        .index
        .tolist()
    )

    fig, axes = plt.subplots(len(selected_numeric), 1, figsize=(10, 4 * len(selected_numeric)))
    if len(selected_numeric) == 1:
        axes = [axes]
    for ax, col in zip(axes, selected_numeric):
        sns.histplot(df[col], kde=True, ax=ax, color='#1f77b4')
        ax.axvline(df[col].mean(), color='red', linestyle='--', alpha=0.6, label='media')
        ax.legend(loc='upper right')
        ax.set_title(f'Distribución de {col}')
    plt.tight_layout()
else:
    print('Sin variables numéricas para graficar.')


In [ ]:
# -----------------------------------------------------------
# Comportamiento y posibles outliers del objetivo UITI
# -----------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[TARGET_COL], kde=True, ax=axes[0], color='#d62728')
axes[0].set_title('Distribución UITI')
sns.boxplot(x=df[TARGET_COL], ax=axes[1], color='#ff9896')
axes[1].set_title('Boxplot UITI')
plt.tight_layout()

display(df[TARGET_COL].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]))


In [ ]:
# -----------------------------------------------------------
# Frecuencias top de variables categóricas para entender densidad operacional
# -----------------------------------------------------------
if categorical_features:
    for col in categorical_features:
        freq_df = df[col].value_counts().head(10).reset_index().rename(columns={'index': col, col: 'conteo'})
        fig = px.bar(freq_df, x=col, y='conteo', title=f'Distribución top 10 de {col}')
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()
else:
    print('No se detectaron columnas categóricas.')


In [ ]:
# -----------------------------------------------------------
# Correlaciones numéricas enfocadas en UITI
# -----------------------------------------------------------
if numeric_features:
    corr_matrix = df[numeric_features + [TARGET_COL]].corr()
    plt.figure(figsize=(11, 8))
    sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False)
    plt.title('Matriz de correlación (numéricas + UITI)')
    plt.show()

    target_corr = (
        corr_matrix[TARGET_COL]
        .drop(TARGET_COL)
        .abs()
        .sort_values(ascending=False)
        .head(min(5, len(numeric_features)))
    )
    display(target_corr.to_frame('corr_abs_uiti'))
else:
    print('No se puede calcular correlación sin variables numéricas.')


In [ ]:
# -----------------------------------------------------------
# Relaciones bilaterales entre UITI y las features más correlacionadas
# -----------------------------------------------------------
if numeric_features:
    corr_matrix = df[numeric_features + [TARGET_COL]].corr()
    main_drivers = (
        corr_matrix[TARGET_COL]
        .drop(TARGET_COL)
        .abs()
        .sort_values(ascending=False)
        .head(min(3, len(numeric_features)))
        .index
        .tolist()
    )

    fig, axes = plt.subplots(len(main_drivers), 1, figsize=(10, 4 * len(main_drivers)))
    if len(main_drivers) == 1:
        axes = [axes]
    for ax, col in zip(axes, main_drivers):
        sns.regplot(x=df[col], y=df[TARGET_COL], ax=ax, scatter_kws={'alpha': 0.25})
        ax.set_title(f'UITI vs {col}')
    plt.tight_layout()
else:
    print('Sin variables numéricas para scatter plots.')


## Ingeniería NFL sobre el dataset
- Derivamos componentes temporales (año, mes, día, día de la semana) para cada fecha fiable.
- Transformamos magnitudes largas (duraciones/intensidades) con `log1p` + `zscore`.
- Añadimos frecuencias de aparición para códigos/ids/categorías.
- Colapsamos categorías raras bajo la etiqueta `OTROS_NFL` para estabilizar el one-hot.


In [ ]:
# -----------------------------------------------------------
# Construcción del dataframe enriquecido con nuevas features
# -----------------------------------------------------------
df_fe = df.copy()
feature_notes = []

parsed_datetime_cols = []
for col in datetime_features:
    parsed = pd.to_datetime(df_fe[col], errors='coerce', infer_datetime_format=True)
    coverage = parsed.notna().mean()
    if coverage < 0.5:
        continue
    df_fe[col] = parsed
    df_fe[f'{col}_year'] = parsed.dt.year
    df_fe[f'{col}_month'] = parsed.dt.month
    df_fe[f'{col}_day'] = parsed.dt.day
    df_fe[f'{col}_dow'] = parsed.dt.dayofweek
    parsed_datetime_cols.append(col)
    feature_notes.extend([f'{col}_year', f'{col}_month', f'{col}_day', f'{col}_dow'])

possible_duration_cols = [col for col in df_fe.columns if 'dur' in col.lower() or 'tiempo' in col.lower()]
for col in possible_duration_cols:
    df_fe[col] = pd.to_numeric(df_fe[col], errors='coerce')
    df_fe[f'{col}_log1p'] = np.log1p(df_fe[col].clip(lower=0))
    df_fe[f'{col}_zscore'] = (df_fe[col] - df_fe[col].mean()) / (df_fe[col].std() or 1)
    feature_notes.extend([f'{col}_log1p', f'{col}_zscore'])

cat_for_freq = sorted(set(categorical_features + [c for c in id_like_features if c in df.columns]))
for col in cat_for_freq:
    freq_series = df_fe[col].value_counts(dropna=False)
    df_fe[f'{col}_freq'] = df_fe[col].map(freq_series).fillna(0)
    feature_notes.append(f'{col}_freq')


def collapse_rare_categories(series: pd.Series, min_freq: float = 0.01) -> pd.Series:
    freq = series.value_counts(normalize=True, dropna=False)
    rare_labels = freq[freq < min_freq].index
    return series.replace(rare_labels, 'OTROS_NFL')

for col in categorical_features:
    df_fe[col] = collapse_rare_categories(df_fe[col].astype('string'))

numeric_features_enhanced = [col for col in df_fe.select_dtypes(include=[np.number]).columns if col != TARGET_COL]
categorical_features_clean = [col for col in df_fe.select_dtypes(exclude=[np.number]).columns if col != TARGET_COL]

print(f'Columnas originales: {df.shape[1]} -> tras ingeniería: {df_fe.shape[1]}')
print('Nuevas columnas clave:', feature_notes[:15], '...')


In [ ]:
# -----------------------------------------------------------
# Pipeline de preprocesamiento + splits listos para DL
# -----------------------------------------------------------
numeric_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('yeojohnson', PowerTransformer(method='yeo-johnson', standardize=False)),
        ('scaler', StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=False)),
    ]
)

transformers = []
if numeric_features_enhanced:
    transformers.append(('numeric', numeric_pipeline, numeric_features_enhanced))
if categorical_features_clean:
    transformers.append(('categorical', categorical_pipeline, categorical_features_clean))

preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

X = df_fe.drop(columns=[TARGET_COL])
y = df_fe[TARGET_COL]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

preprocessor.fit(X_train)
X_train_ready = preprocessor.transform(X_train)
X_valid_ready = preprocessor.transform(X_valid)

if hasattr(preprocessor, 'get_feature_names_out'):
    feature_names_out = preprocessor.get_feature_names_out().tolist()
else:
    feature_names_out = numeric_features_enhanced
    if categorical_features_clean:
        encoder = preprocessor.named_transformers_['categorical'].named_steps['encoder']
        feature_names_out += encoder.get_feature_names_out(categorical_features_clean).tolist()

print(f'X_train listo: {X_train_ready.shape} | X_valid listo: {X_valid_ready.shape}')
print(f'Número total de features tras preprocesar: {len(feature_names_out)}')


In [ ]:
# -----------------------------------------------------------
# Conversión a tf.data datasets para acelerar el entrenamiento NFL
# -----------------------------------------------------------
import numpy.typing as npt

def build_tf_dataset(features: npt.NDArray, labels: pd.Series, batch_size: int = 256, shuffle: bool = True):
    if tf is None:
        raise ImportError('TensorFlow no está disponible en este entorno.')
    dataset = tf.data.Dataset.from_tensor_slices((features.astype('float32'), labels.to_numpy(dtype='float32')))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(features), reshuffle_each_iteration=True)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

if tf is not None:
    train_ds = build_tf_dataset(X_train_ready, y_train)
    valid_ds = build_tf_dataset(X_valid_ready, y_valid, shuffle=False)
    print(train_ds)
    print(valid_ds)
else:
    print('Instala tensorflow para materializar tf.data datasets.')


In [ ]:
# -----------------------------------------------------------
# Plantilla base de red neuronal para UITI (ajusta hiperparámetros NFL)
# -----------------------------------------------------------
from typing import Sequence

def build_nfl_regressor(input_dim: int, hidden_units: Sequence[int] = (256, 128, 64), dropout_rate: float = 0.15):
    if tf is None:
        raise ImportError('TensorFlow no está instalado; no se puede construir la red.')

    inputs = tf.keras.Input(shape=(input_dim,), name='feature_vector')
    x = inputs
    for units in hidden_units:
        x = tf.keras.layers.Dense(units, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(1, activation='linear', name='uiti')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='nfl_uiti_model')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='mse',
        metrics=[tf.keras.metrics.MeanAbsoluteError(name='mae')],
    )
    return model

if tf is not None and 'X_train_ready' in globals():
    nfl_model = build_nfl_regressor(X_train_ready.shape[1])
    nfl_model.summary()
else:
    print('Activa TensorFlow para instanciar y resumir el modelo NFL.')
